In [1]:
import os
os.environ["PYTHONWARNINGS"] = "ignore"

In [2]:
import warnings
warnings.filterwarnings("ignore")

In [3]:
import spatialdata_io as io
import shutil
import glob
import json

In [ ]:
# https://cf.10xgenomics.com/samples/xenium/3.0.0/Xenium_Prime_Breast_Cancer_FFPE/Xenium_Prime_Breast_Cancer_FFPE_analysis_summary.html

In [ ]:
xenium_path = "breast_data/"
zarr_path = "breast_data/xenium_breast.zarr"

In [ ]:
sdata = io.xenium(xenium_path)

In [6]:
shutil.rmtree(zarr_path)

In [7]:
sdata.write(zarr_path, overwrite=True)

In [10]:
def _regular_chunk_shape(dim_specs):
    """Extract the uniform chunk size per dim from a rectilinear spec (ignores the partial tail)."""
    result = []
    for dim in dim_specs:
        first = dim[0]
        result.append(first[0] if isinstance(first, list) else first)
    return result

def fix_rectilinear_to_regular(zarr_store_path):
    """
    Convert rectilinear chunk grids → regular chunk grids in a zarr v3 store.
    Only the JSON metadata is rewritten; chunk data files are untouched.
    """
    def convert(cg):
        if cg.get("name") != "rectilinear":
            return cg
        chunk_shape = _regular_chunk_shape(cg["configuration"]["chunk_shapes"])
        return {"name": "regular", "configuration": {"chunk_shape": chunk_shape}}

    fixed = 0
    for path in sorted(glob.glob(f"{zarr_store_path}/**/*.json", recursive=True)):
        d = json.load(open(path))
        if d.get("chunk_grid", {}).get("name") == "rectilinear":
            d["chunk_grid"] = convert(d["chunk_grid"])
            with open(path, "w") as fh:
                json.dump(d, fh)
            fixed += 1

    # Also update the consolidated metadata embedded in the root zarr.json
    root_path = f"{zarr_store_path}/zarr.json"
    root = json.load(open(root_path))
    for node in root.get("consolidated_metadata", {}).get("metadata", {}).values():
        if isinstance(node, dict) and node.get("chunk_grid", {}).get("name") == "rectilinear":
            node["chunk_grid"] = convert(node["chunk_grid"])
    with open(root_path, "w") as fh:
        json.dump(root, fh)

    print(f"Converted {fixed} rectilinear chunk grids → regular in {zarr_store_path}")

In [9]:
fix_rectilinear_to_regular(zarr_path)

Converted 0 rectilinear chunk grids → regular in data/xenium_breast.zarr
